# Load Built-in Dataset

In [7]:
from surprise import Dataset, Reader

# Path to the u.data file
file_path = r"C:\Users\pouya\Downloads\Recommender_system-master\Recommender_system-master\dataset\dataset\ml-100k\u.data"

# Define the format (user, item, rating, timestamp)
reader = Reader(line_format='user item rating timestamp', sep='\t')

# Load the dataset
data = Dataset.load_from_file(file_path, reader=reader)

type(data)

surprise.dataset.DatasetAutoFolds

# Data Splitting & Exploration

In [117]:
from surprise.model_selection import train_test_split

# 80% train, 20% test
trainset, testset = train_test_split(data=data,
                                         test_size=0.2,
                                         random_state=43)

# Here trainset is a Trainset object but testset is a list
print(f"Train data: {trainset.n_ratings}")
print(f"Test data: {len(testset)}")
print(f"Train users: {trainset.n_users}")
print(f"Train items: {trainset.n_items}")

Train data: 80000
Test data: 20000
Train users: 943
Train items: 1645


## Model Training: SVD (Singular Value Decomposition)

In [ ]:
from surprise import SVD

# Initialize SVD model
svd_model = SVD(n_factors=100,  # latent factros (number of hidden features)
                n_epochs=20,    # Training iterations
                lr_all=0.005,   # Learning rate
                reg_all=0.02)   # Regularization

# Train on 80% of data
svd_model.fit(trainset)

## Making Predictions with the Trained SVD Model


In [142]:
svd_model.predict(uid="10", iid="196"), svd_model.predict(uid="25", iid="365")

(Prediction(uid='10', iid='196', r_ui=None, est=4.382611053063427, details={'was_impossible': False}),
 Prediction(uid='25', iid='365', r_ui=None, est=3.6061974792246296, details={'was_impossible': False}))

## Model Evaluation


In [ ]:
from surprise import accuracy

# Generate predictions on test set (unseen data)
predictions = svd_model.test(testset)

# Calculate evaluations metrics
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

print(rmse, mae)

RMSE: 0.9432
MAE:  0.7433
0.9432400936965007 0.7432878547207737


## Hyperparameter Tuning with GridSearchCV


In [ ]:
from surprise.model_selection import GridSearchCV

# Define hyperparameter grid to search
param_grid = {
    "n_factors": [50, 100, 150],
    "n_epochs": [20, 30],
    "lr_all": [0.005, 0.01, 0.001],
    "reg_all":[0.02, 0.1]
}

# 3-fold cross-validation to find the best parameters
gs = GridSearchCV(algo_class=SVD,
                  param_grid=param_grid,
                  measures=["rmse", "mae"], 
                  cv=3)

# Run search on all data
gs.fit(data)

print(f"Best RMSE score: {gs.best_score['rmse']}")
print(f"Best RMSE score: {gs.best_score['mae']}")
print(f"Best Parameters: {gs.best_params['rmse']}")

Best RMSE score: 0.9224802183813177
Best RMSE score: 0.7299294954702837
Best Parameters: {'n_factors': 100, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}


## Training Final Model with The Best Parameters

In [ ]:
# Unpack best parameters and initialize SVD
best_svd_model = SVD(**gs.best_params["rmse"])
best_svd_model.fit(trainset)

## Making Predictions with the Trained Model

In [143]:
best_svd_model.predict(uid="10", iid="196"), best_svd_model.predict(uid="25", iid="365")

(Prediction(uid='10', iid='196', r_ui=None, est=4.184385708116004, details={'was_impossible': False}),
 Prediction(uid='25', iid='365', r_ui=None, est=3.4353843993906423, details={'was_impossible': False}))

## Finding High-Rated Predictions (Users & Items 1-100)

In [186]:
high_ratings = []

# First 100 users and items
users = range(1, 101)
items = range(1, 101)

# Loop through all users-items combinations and keep only >=4.7 stars
for user in users:
    for item in items:
        pred = best_svd_model.predict(str(user), str(item))
        if pred.est >= 4.7:
            high_ratings.append((user, item, pred.est))

print(f"Found {len(high_ratings)} predictions above 4.7! \n")

for user, item, est in high_ratings:
    print(f"User {user} → Item {item}: {est:.2f}")

# shows the unique users and items
print(f"\nUnique users: {len(set([user for user, item, rate in high_ratings]))}")
print(f"Unique items: {len(set([item for user, item, rate in high_ratings]))}")

Found 102 predictions above 4.7! 

User 4 → Item 8: 4.76
User 4 → Item 9: 4.89
User 4 → Item 10: 4.78
User 4 → Item 12: 5.00
User 4 → Item 14: 4.97
User 4 → Item 19: 4.74
User 4 → Item 23: 4.89
User 4 → Item 28: 4.75
User 4 → Item 30: 5.00
User 4 → Item 32: 4.73
User 4 → Item 45: 4.87
User 4 → Item 46: 4.78
User 4 → Item 48: 5.00
User 4 → Item 50: 4.89
User 4 → Item 52: 4.80
User 4 → Item 56: 4.94
User 4 → Item 57: 4.84
User 4 → Item 59: 5.00
User 4 → Item 60: 5.00
User 4 → Item 61: 4.77
User 4 → Item 64: 5.00
User 4 → Item 79: 4.72
User 4 → Item 83: 4.87
User 4 → Item 86: 4.95
User 4 → Item 87: 4.84
User 4 → Item 89: 5.00
User 4 → Item 98: 5.00
User 4 → Item 100: 5.00
User 7 → Item 12: 4.80
User 7 → Item 50: 4.72
User 7 → Item 64: 4.83
User 7 → Item 98: 4.73
User 8 → Item 50: 4.75
User 8 → Item 64: 4.80
User 9 → Item 12: 4.93
User 9 → Item 50: 4.72
User 9 → Item 64: 4.73
User 10 → Item 64: 4.74
User 12 → Item 22: 4.71
User 12 → Item 64: 5.00
User 16 → Item 12: 4.99
User 16 → Item 22: 

## Evaluating Model on Test Set


In [ ]:
print("First 10 testset ratings (user, item, rating):\n")

for i, (user, item, rating) in enumerate(testset[:10]):
    pred = round(best_svd_model.predict(str(user), str(item)).est, 2)
    print(f"User {user} → Item {item}: Real: {rating} → Predicted:{pred}")

First 10 testset ratings (user, item, rating):

User 160 → Item 150: Real: 4.0 → Predicted:4.16
User 95 → Item 65: Real: 4.0 → Predicted:3.35
User 347 → Item 97: Real: 4.0 → Predicted:3.94
User 927 → Item 763: Real: 4.0 → Predicted:3.78
User 747 → Item 403: Real: 5.0 → Predicted:3.68
User 265 → Item 591: Real: 5.0 → Predicted:3.65
User 883 → Item 277: Real: 4.0 → Predicted:3.69
User 682 → Item 298: Real: 4.0 → Predicted:3.58
User 233 → Item 521: Real: 5.0 → Predicted:4.04
User 456 → Item 743: Real: 2.0 → Predicted:2.46
